In [1]:
!git clone https://github.com/fastapi/fastapi.git

fatal: destination path 'fastapi' already exists and is not an empty directory.


In [2]:
!ls

data  fastapi  sample_data


In [3]:
%cd fastapi

/content/fastapi


In [4]:
!git log --oneline -5

255b91292 (HEAD -> master, tag: 0.140.0, origin/master, origin/HEAD) 🔖 Release version 0.140.0 (#16050)
892eacd27 📝 Update release notes
027082950 ⚡️ Reduce memory usage in dependencies (#16049)
ae031be7b 📝 Update release notes
f3644b33c 📝 Fix links in docs (#15967)


In [5]:
%cd

/root


In [6]:
!pip install -q \
langchain \
langchain-community \
langchain-google-genai \
google-generativeai \
sentence-transformers \
faiss-cpu \
GitPython \
ragas

In [7]:
import git
import faiss
import langchain
import requests
import google.genai

print("Setup Successful ✅")

Setup Successful ✅


In [8]:
import os

folders = [
    "data/raw",
    "data/processed",
    "data/faiss_index",
    "ingestion",
    "embeddings",
    "retrieval",
    "guard",
    "evaluation",
    "utils"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created!")

Folders created!


In [9]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [10]:
!git clone https://github.com/fastapi/fastapi.git

fatal: destination path 'fastapi' already exists and is not an empty directory.


In [11]:
%cd fastapi

/root/fastapi


In [12]:
import git
import pandas as pd
from tqdm import tqdm

In [13]:
repo = git.Repo("/content/fastapi")
print(repo)

<git.repo.base.Repo '/content/fastapi/.git'>


In [14]:
commits = []

for commit in tqdm(repo.iter_commits()):
    commits.append({
        "sha": commit.hexsha,
        "author": commit.author.name,
        "email": commit.author.email,
        "date": commit.committed_datetime,
        "message": commit.message.strip()
    })

print(f"Total commits: {len(commits)}")

7538it [00:01, 5305.26it/s]

Total commits: 7538


In [15]:
commit_df = pd.DataFrame(commits)

commit_df.head()

,sha,author,email,date,message
0,255b912928904e3ba5980425a54d6837c8bd1a1c,Sebastián Ramírez,tiangolo@gmail.com,2026-07-24 21:15:37+00:00,🔖 Release version 0.140.0 (#16050)\n\nCo-autho...
1,892eacd27dcdcfe6c5ea114543ff473cbbfd7c6f,github-actions[bot],github-actions[bot]@users.noreply.github.com,2026-07-24 21:08:24+00:00,📝 Update release notes\n\n[skip ci]
2,027082950068d6e3897b5422ab6cde3168f7d8b0,Sebastián Ramírez,tiangolo@gmail.com,2026-07-24 21:07:51+00:00,⚡️ Reduce memory usage in dependencies (#16049)
3,ae031be7b5a2df589f63f4d25389cf07b7b8fc86,github-actions[bot],github-actions[bot]@users.noreply.github.com,2026-07-24 20:30:21+00:00,📝 Update release notes\n\n[skip ci]
4,f3644b33cdaf1b14868d77f1c309e0a455c6bac8,Yurii Motov,109919500+YuriiMotov@users.noreply.github.com,2026-07-24 20:29:32+00:00,📝 Fix links in docs (#15967)\n\nCo-authored-by...


In [16]:
os.makedirs("/content/data/raw", exist_ok=True)

commit_df.to_csv("/content/data/raw/commits.csv", index=False)

print("Commits saved successfully!")

Commits saved successfully!


In [17]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

In [18]:
OWNER = "fastapi"
REPO = "fastapi"

headers = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {GITHUB_TOKEN}"
}

In [19]:
prs = []

page = 1

while True:
    url = f"https://api.github.com/repos/{OWNER}/{REPO}/pulls"

    params = {
        "state": "all",
        "per_page": 100,
        "page": page
    }

    response = requests.get(url, headers=headers, params=params)

    data = response.json()

    if len(data) == 0:
        break

    prs.extend(data)

    print(f"Fetched page {page} ({len(data)} PRs)")

    page += 1

Fetched page 1 (100 PRs)
Fetched page 2 (100 PRs)
Fetched page 3 (100 PRs)
Fetched page 4 (100 PRs)
Fetched page 5 (100 PRs)
Fetched page 6 (100 PRs)
Fetched page 7 (100 PRs)
Fetched page 8 (100 PRs)
Fetched page 9 (100 PRs)
Fetched page 10 (100 PRs)
Fetched page 11 (100 PRs)
Fetched page 12 (100 PRs)
Fetched page 13 (100 PRs)
Fetched page 14 (100 PRs)
Fetched page 15 (100 PRs)
Fetched page 16 (100 PRs)
Fetched page 17 (100 PRs)
Fetched page 18 (100 PRs)
Fetched page 19 (100 PRs)
Fetched page 20 (100 PRs)
Fetched page 21 (100 PRs)
Fetched page 22 (100 PRs)
Fetched page 23 (100 PRs)
Fetched page 24 (100 PRs)
Fetched page 25 (100 PRs)
Fetched page 26 (100 PRs)
Fetched page 27 (100 PRs)
Fetched page 28 (100 PRs)
Fetched page 29 (100 PRs)
Fetched page 30 (100 PRs)
Fetched page 31 (100 PRs)
Fetched page 32 (100 PRs)
Fetched page 33 (100 PRs)
Fetched page 34 (100 PRs)
Fetched page 35 (100 PRs)
Fetched page 36 (100 PRs)
Fetched page 37 (100 PRs)
Fetched page 38 (100 PRs)
Fetched page 39 (100 

In [20]:
pr_records = []

for pr in prs:
    pr_records.append({
        "id": pr["id"],
        "number": pr["number"],
        "title": pr["title"],
        "body": pr["body"],
        "state": pr["state"],
        "created_at": pr["created_at"],
        "merged_at": pr["merged_at"],
        "user": pr["user"]["login"],
        "html_url": pr["html_url"]
    })

In [21]:
pr_df = pd.DataFrame(pr_records)

pr_df.head()

,id,number,title,body,state,created_at,merged_at,user,html_url
0,4135652912,16060,Add security tip about rate limiting and brute...,Added a security tip to the Security - First S...,open,2026-07-26T09:00:45Z,None,Dev9269,https://github.com/fastapi/fastapi/pull/16060
1,4134232202,16058,📝 Update ReDoc URLs from Rebilly/ReDoc to Redo...,Closes #15432\n\nReDoc repository was moved fr...,closed,2026-07-25T22:56:07Z,None,adnanahamed66772ndpc,https://github.com/fastapi/fastapi/pull/16058
2,4131699095,16055,Fix SSE data payload dropping trailing newlines,### Summary\n\n`format_sse_event` splits the d...,closed,2026-07-25T08:12:44Z,None,vidigoat,https://github.com/fastapi/fastapi/pull/16055
3,4130738184,16054,feat(utils): add truncate_middle helper functi...,## Description\nAdds a type-annotated \\trunca...,closed,2026-07-25T02:51:54Z,None,mahenoorsalat,https://github.com/fastapi/fastapi/pull/16054
4,4129313995,16050,🔖 Release version 0.140.0,Prepare release 0.140.0.,closed,2026-07-24T21:09:36Z,2026-07-24T21:15:38Z,tiangolo,https://github.com/fastapi/fastapi/pull/16050


In [22]:
os.makedirs("/content/data/raw", exist_ok=True)
pr_df.to_csv("/content/data/raw/pull_requests.csv", index=False)
print("Pull requests saved successfully!")

Pull requests saved successfully!


In [23]:
issues = []

page = 1

while True:
    url = f"https://api.github.com/repos/{OWNER}/{REPO}/issues"

    params = {
        "state": "all",
        "per_page": 100,
        "page": page
    }

    response = requests.get(url, headers=headers, params=params)

    data = response.json()

    if len(data) == 0:
        break

    # Keep only actual issues (exclude pull requests)
    actual_issues = [item for item in data if "pull_request" not in item]

    issues.extend(actual_issues)

    print(f"Fetched page {page} ({len(actual_issues)} issues)")

    page += 1

Fetched page 1 (5 issues)
Fetched page 2 (3 issues)
Fetched page 3 (9 issues)
Fetched page 4 (2 issues)
Fetched page 5 (5 issues)
Fetched page 6 (4 issues)
Fetched page 7 (6 issues)
Fetched page 8 (5 issues)
Fetched page 9 (4 issues)
Fetched page 10 (3 issues)
Fetched page 11 (1 issues)
Fetched page 12 (0 issues)
Fetched page 13 (19 issues)
Fetched page 14 (5 issues)
Fetched page 15 (5 issues)
Fetched page 16 (2 issues)
Fetched page 17 (2 issues)
Fetched page 18 (2 issues)
Fetched page 19 (3 issues)
Fetched page 20 (3 issues)
Fetched page 21 (3 issues)
Fetched page 22 (11 issues)
Fetched page 23 (2 issues)
Fetched page 24 (2 issues)
Fetched page 25 (0 issues)
Fetched page 26 (2 issues)
Fetched page 27 (13 issues)
Fetched page 28 (5 issues)
Fetched page 29 (7 issues)
Fetched page 30 (8 issues)
Fetched page 31 (9 issues)
Fetched page 32 (10 issues)
Fetched page 33 (11 issues)
Fetched page 34 (2 issues)
Fetched page 35 (7 issues)
Fetched page 36 (4 issues)
Fetched page 37 (8 issues)
Fetch

In [24]:
print(f"Total Issues: {len(issues)}")

Total Issues: 3543


In [25]:
issue_records = []

for issue in issues:
    issue_records.append({
        "id": issue["id"],
        "number": issue["number"],
        "title": issue["title"],
        "body": issue["body"],
        "state": issue["state"],
        "created_at": issue["created_at"],
        "closed_at": issue["closed_at"],
        "user": issue["user"]["login"],
        "comments": issue["comments"],
        "labels": ", ".join([label["name"] for label in issue["labels"]]),
        "html_url": issue["html_url"]
    })

In [26]:
issue_df = pd.DataFrame(issue_records)

issue_df.head()

,id,number,title,body,state,created_at,closed_at,user,comments,labels,html_url
0,4973000181,16053,Add REFUTE scientific critique + calibration b...,## Proposal\n\nAdd **REFUTE** to related evalu...,closed,2026-07-24T23:32:46Z,2026-07-25T03:46:16Z,connerlambden,1,,https://github.com/fastapi/fastapi/issues/16053
1,4956219102,16037,Type annotation mismatch in sub-dependencies e...,### Privileged issue\n\n- [x] I'm @tiangolo or...,closed,2026-07-23T06:47:42Z,2026-07-23T07:46:53Z,SijanMahmudAI,1,,https://github.com/fastapi/fastapi/issues/16037
2,4900328985,16010,"app.frontend() with fallback=""index.html"" retu...",\n### Discussed in https://github.com/fastapi/...,closed,2026-07-16T08:50:06Z,2026-07-16T09:21:25Z,tiangolo,1,bug,https://github.com/fastapi/fastapi/issues/16010
3,4864140127,15974,Race condition in _IncludedRouter cache rebuil...,### Description\n\nSince the router refactor i...,closed,2026-07-11T20:58:48Z,2026-07-12T04:47:50Z,UditDewan,0,,https://github.com/fastapi/fastapi/issues/15974
4,4852026509,15969,Tracking: Current open issues (links to Roadmap),## Current Open Issues Summary\n\n**As of 2026...,closed,2026-07-10T04:34:05Z,2026-07-10T04:38:36Z,pi-prakhar,1,,https://github.com/fastapi/fastapi/issues/15969


In [27]:
os.makedirs("/content/data/raw", exist_ok=True)

issue_df.to_csv("/content/data/raw/issues.csv", index=False)

print("Issues saved successfully!")

Issues saved successfully!


In [28]:
def get_pr_comments(pr_number):
    comments = []
    page = 1

    while True:
        url = f"https://api.github.com/repos/{OWNER}/{REPO}/issues/{pr_number}/comments"

        response = requests.get(
            url,
            headers=headers,
            params={"per_page": 100, "page": page}
        )

        if response.status_code != 200:
            print(f"Error {response.status_code} for PR {pr_number}")
            print(response.json())
            break

        data = response.json()

        if not isinstance(data, list):
            print(f"Unexpected response for PR {pr_number}")
            print(data)
            break

        if len(data) == 0:
            break

        comments.extend(data)
        page += 1

    return comments

In [29]:
def get_review_comments(pr_number):
    comments = []
    page = 1

    while True:
        url = f"https://api.github.com/repos/{OWNER}/{REPO}/pulls/{pr_number}/comments"

        response = requests.get(
            url,
            headers=headers,
            params={
                "per_page": 100,
                "page": page
            }
        )

        if response.status_code != 200:
            print(f"Error fetching review comments for PR #{pr_number}")
            print(response.json())
            break

        data = response.json()

        if not isinstance(data, list):
            print(f"Unexpected response for PR #{pr_number}")
            break

        if len(data) == 0:
            break

        comments.extend(data)

        page += 1

        time.sleep(0.1)

    return comments

In [30]:
from tqdm import tqdm

pr_documents = []

# adding limit  for fast testing:
pr_records = pr_records[-200:]

for pr in tqdm(pr_records):

    try:
        issue_comments = get_pr_comments(pr["number"])
        review_comments = get_review_comments(pr["number"])

        issue_text = "\n".join(
            comment["body"]
            for comment in issue_comments
            if isinstance(comment, dict) and comment.get("body")
        )

        review_text = "\n".join(
            comment["body"]
            for comment in review_comments
            if isinstance(comment, dict) and comment.get("body")
        )

        full_text = f"""
PR #{pr['number']}

Title:
{pr['title']}

Description:
{pr.get('body') or ''}

Issue Discussion:
{issue_text}

Review Discussion:
{review_text}
"""

        pr_documents.append({
            "number": pr["number"],
            "title": pr["title"],
            "text": full_text,
            "url": pr["html_url"]
        })

    except Exception as e:
        print(f"Skipped PR #{pr['number']} because of error: {e}")

  0%|          | 1/200 [00:01<04:01,  1.21s/it]

Skipped PR #568 because of error: name 'time' is not defined


  2%|▏         | 3/200 [00:03<03:54,  1.19s/it]

Skipped PR #556 because of error: name 'time' is not defined


 12%|█▏        | 24/200 [00:26<03:22,  1.15s/it]

...

 81%|████████  | 162/200 [02:59<00:43,  1.16s/it]

Skipped PR #92 because of error: name 'time' is not defined


 98%|█████████▊| 197/200 [03:36<00:03,  1.11s/it]

Skipped PR #20 because of error: name 'time' is not defined


100%|██████████| 200/200 [03:39<00:00,  1.10s/it]


In [31]:
import json

with open("/content/data/raw/pr_documents.json","w",encoding="utf-8") as f:
    json.dump(pr_documents,f,indent=4,ensure_ascii=False)

In [32]:
def get_issue_comments(issue_number):
    comments = []
    page = 1

    while True:
        url = f"https://api.github.com/repos/{OWNER}/{REPO}/issues/{issue_number}/comments"

        response = requests.get(
            url,
            headers=headers,
            params={
                "per_page": 100,
                "page": page
            }
        )

        if response.status_code != 200:
            print(f"Error fetching Issue #{issue_number}")
            break

        data = response.json()

        if not isinstance(data, list):
            break

        if len(data) == 0:
            break

        comments.extend(data)

        page += 1

    return comments

In [33]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU Name: Tesla T4


In [34]:
from tqdm import tqdm

issue_documents = []


issue_records = issue_records[-200:]

for issue in tqdm(issue_records):

    comments = get_issue_comments(issue["number"])

    discussion = "\n".join(
        c["body"]
        for c in comments
        if isinstance(c, dict) and c.get("body")
    )

    text = f"""
Issue #{issue['number']}

Title:
{issue['title']}

Description:
{issue.get('body') or ''}

Discussion:
{discussion}
"""

    issue_documents.append({
        "number": issue["number"],
        "title": issue["title"],
        "text": text,
        "url": issue["html_url"]
    })

100%|██████████| 200/200 [02:38<00:00,  1.26it/s]


In [35]:
import json

with open("/content/data/raw/issue_documents.json", "w", encoding="utf-8") as f:
    json.dump(issue_documents, f, indent=4, ensure_ascii=False)

print(f"Saved {len(issue_documents)} issue documents.")

Saved 200 issue documents.


In [36]:
print(f"Total PRs: {len(pr_documents)}")
print(f"Total Issues: {len(issue_documents)}")

Total PRs: 179
Total Issues: 200


In [37]:
pr_df = pd.DataFrame(pr_documents)
issue_df = pd.DataFrame(issue_documents)

pr_df.head()

,number,title,text,url
0,557,Add documentation for static swagger (#112),\nPR #557\n\nTitle:\nAdd documentation for sta...,https://github.com/fastapi/fastapi/pull/557
1,554,Alphabetically sort schemas,\nPR #554\n\nTitle:\nAlphabetically sort schem...,https://github.com/fastapi/fastapi/pull/554
2,547,fix bug with 4xx check,\nPR #547\n\nTitle:\nfix bug with 4xx check\n\...,https://github.com/fastapi/fastapi/pull/547
3,538,Preserve route_class when calling include_router,\nPR #538\n\nTitle:\nPreserve route_class when...,https://github.com/fastapi/fastapi/pull/538
4,537,fix: wrong doctype in docs html,\nPR #537\n\nTitle:\nfix: wrong doctype in doc...,https://github.com/fastapi/fastapi/pull/537


In [38]:
issue_df.head()

,number,title,text,url
0,337,Is the document old or bugs?,\nIssue #337\n\nTitle:\nIs the document old or...,https://github.com/fastapi/fastapi/issues/337
1,336,How to doc a path with only optional queries?,\nIssue #336\n\nTitle:\nHow to doc a path with...,https://github.com/fastapi/fastapi/issues/336
2,335,Tutorial on Authorization Code Grant Flow,\nIssue #335\n\nTitle:\nTutorial on Authorizat...,https://github.com/fastapi/fastapi/issues/335
3,334,Large StreamingResponse locks up server,\nIssue #334\n\nTitle:\nLarge StreamingRespons...,https://github.com/fastapi/fastapi/issues/334
4,332,OpenAPI schema generation sometimes fails,\nIssue #332\n\nTitle:\nOpenAPI schema generat...,https://github.com/fastapi/fastapi/issues/332


In [39]:
from langchain_core.documents import Document

In [40]:
commit_df = pd.read_csv("/content/data/raw/commits.csv")

print(commit_df.shape)
commit_df.head()

(7538, 5)


,sha,author,email,date,message
0,255b912928904e3ba5980425a54d6837c8bd1a1c,Sebastián Ramírez,tiangolo@gmail.com,2026-07-24 21:15:37+00:00,🔖 Release version 0.140.0 (#16050)\n\nCo-autho...
1,892eacd27dcdcfe6c5ea114543ff473cbbfd7c6f,github-actions[bot],github-actions[bot]@users.noreply.github.com,2026-07-24 21:08:24+00:00,📝 Update release notes\n\n[skip ci]
2,027082950068d6e3897b5422ab6cde3168f7d8b0,Sebastián Ramírez,tiangolo@gmail.com,2026-07-24 21:07:51+00:00,⚡️ Reduce memory usage in dependencies (#16049)
3,ae031be7b5a2df589f63f4d25389cf07b7b8fc86,github-actions[bot],github-actions[bot]@users.noreply.github.com,2026-07-24 20:30:21+00:00,📝 Update release notes\n\n[skip ci]
4,f3644b33cdaf1b14868d77f1c309e0a455c6bac8,Yurii Motov,109919500+YuriiMotov@users.noreply.github.com,2026-07-24 20:29:32+00:00,📝 Fix links in docs (#15967)\n\nCo-authored-by...


In [41]:
with open("/content/data/raw/pr_documents.json", "r", encoding="utf-8") as f:
    pr_documents = json.load(f)

print(len(pr_documents))

179


In [42]:
with open("/content/data/raw/issue_documents.json", "r", encoding="utf-8") as f:
    issue_documents = json.load(f)

print(len(issue_documents))

200


In [43]:
documents = []

for _, row in commit_df.iterrows():

    text = f"""
Commit SHA: {row['sha']}

Author:
{row['author']}

Date:
{row['date']}

Commit Message:
{row['message']}
"""

    documents.append(
        Document(
            page_content=text,
            metadata={
                "type": "commit",
                "sha": row["sha"],
                "author": row["author"],
                "date": str(row["date"])
            }
        )
    )

In [44]:
for pr in pr_documents:

    documents.append(
        Document(
            page_content=pr["text"],
            metadata={
                "type": "pull_request",
                "number": pr["number"],
                "url": pr["url"]
            }
        )
    )

In [45]:
for issue in issue_documents:

    documents.append(
        Document(
            page_content=issue["text"],
            metadata={
                "type": "issue",
                "number": issue["number"],
                "url": issue["url"]
            }
        )
    )

In [46]:
print("Total Documents:", len(documents))

Total Documents: 7917


In [47]:
documents[0]

Document(metadata={'type': 'commit', 'sha': '255b912928904e3ba5980425a54d6837c8bd1a1c', 'author': 'Sebastián Ramírez', 'date': '2026-07-24 21:15:37+00:00'}, page_content='\nCommit SHA: 255b912928904e3ba5980425a54d6837c8bd1a1c\n\nAuthor:\nSebastián Ramírez\n\nDate:\n2026-07-24 21:15:37+00:00\n\nCommit Message:\n🔖 Release version 0.140.0 (#16050)\n\nCo-authored-by: github-actions[bot] <github-actions[bot]@users.noreply.github.com>\n')

In [48]:
!pip install -q langchain-text-splitters

In [49]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [50]:
import langchain

print(langchain.__version__)

1.3.13


In [51]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

chunked_documents = text_splitter.split_documents(documents)

print("Original Documents:", len(documents))
print("Chunked Documents:", len(chunked_documents))

Original Documents: 7917
Chunked Documents: 10809


In [52]:
chunked_documents[0]

Document(metadata={'type': 'commit', 'sha': '255b912928904e3ba5980425a54d6837c8bd1a1c', 'author': 'Sebastián Ramírez', 'date': '2026-07-24 21:15:37+00:00'}, page_content='Commit SHA: 255b912928904e3ba5980425a54d6837c8bd1a1c\n\nAuthor:\nSebastián Ramírez\n\nDate:\n2026-07-24 21:15:37+00:00\n\nCommit Message:\n🔖 Release version 0.140.0 (#16050)\n\nCo-authored-by: github-actions[bot] <github-actions[bot]@users.noreply.github.com>')

In [53]:
chunked_documents[0].metadata

{'type': 'commit',
 'sha': '255b912928904e3ba5980425a54d6837c8bd1a1c',
 'author': 'Sebastián Ramírez',
 'date': '2026-07-24 21:15:37+00:00'}

In [54]:
import pickle
import os

os.makedirs("/content/data/processed", exist_ok=True)

with open("/content/data/processed/chunked_documents.pkl", "wb") as f:
    pickle.dump(chunked_documents, f)

print(f"Saved {len(chunked_documents)} chunks successfully.")

Saved 10809 chunks successfully.


In [55]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_6206/3249158374.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings


In [56]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 256
    }
)

/tmp/ipykernel_6206/3914338407.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [57]:
texts = [doc.page_content for doc in chunked_documents]
metadatas = [doc.metadata for doc in chunked_documents]

embeddings = embedding_model.embed_documents(texts)

vector_db = FAISS.from_embeddings(
    text_embeddings=list(zip(texts, embeddings)),
    embedding=embedding_model,
    metadatas=metadatas
)

In [58]:
import os

os.makedirs("/content/data/faiss_index", exist_ok=True)

vector_db.save_local("/content/data/faiss_index")

print("✅ FAISS index saved successfully!")

✅ FAISS index saved successfully!


In [59]:
query = "Why was dependency injection introduced?"

results = vector_db.similarity_search(query, k=5)

for i, doc in enumerate(results, 1):
    print("=" * 80)
    print(f"Result {i}")
    print(doc.metadata)
    print(doc.page_content[:500])

Result 1
{'type': 'issue', 'number': 12, 'url': 'https://github.com/fastapi/fastapi/issues/12'}
Whoa! I should've thought to cite @tiango's docs. Sorry for sending you into the diaper dimension @adamhp . Diapers are cool and all, but Dependency Injection is on another level. Here are the references you seek:

[Getting Started](https://fastapi.tiangolo.com/tutorial/dependencies/)
[Advanced Guide](https://fastapi.tiangolo.com/advanced/advanced-dependencies/)

Hope that helps!
Result 2
{'type': 'commit', 'sha': 'c9758e15a1e6965377009b67b547861bb1971dea', 'author': 'Sebastián Ramírez', 'date': '2018-12-15 18:15:42+04:00'}
Commit SHA: c9758e15a1e6965377009b67b547861bb1971dea

Author:
Sebastián Ramírez

Date:
2018-12-15 18:15:42+04:00

Commit Message:
:memo: Add fist Dependency Injection docs
Result 3
{'type': 'issue', 'number': 81, 'url': 'https://github.com/fastapi/fastapi/issues/81'}
@tiangolo, I'm working with a colleague on a FastAPI application and we're having a discussion about the m

In [60]:
retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 5,
        "fetch_k": 50,
        "filter": {"type": "issue"}
    }
)

In [61]:
query = "How does dependency injection work in FastAPI?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print("=" * 80)
    print(f"Document {i}")
    print(doc.metadata)
    print(doc.page_content[:500])

Document 1
{'type': 'issue', 'number': 212, 'url': 'https://github.com/fastapi/fastapi/issues/212'}
Discussion:
Hey @CRad14, I'm glad to see you chose to learn FastAPI!

## Summary

In an API (or almost in any web application, written in any language or framework), you normally separate the logic of what happens in the sequence of getting a request (sent by the client, let's say a browser with a frontend) and generating the response for that request (that's what your FastAPI app will return to the user).

When your client (e.g. user in a browser) interacts with your API, it sends a singl
Document 2
{'type': 'issue', 'number': 12, 'url': 'https://github.com/fastapi/fastapi/issues/12'}
Whoa! I should've thought to cite @tiango's docs. Sorry for sending you into the diaper dimension @adamhp . Diapers are cool and all, but Dependency Injection is on another level. Here are the references you seek:

[Getting Started](https://fastapi.tiangolo.com/tutorial/dependencies/)
[Advanced Guide](http

In [62]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [63]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0
)

In [64]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are PatchContext, an AI assistant for the FastAPI repository.

Answer the user's question ONLY using the retrieved repository context.

Instructions:
- Combine information from multiple retrieved documents when appropriate.
- Explain the reasoning using only the provided context.
- Do not invent facts.
- If the context is insufficient, say:
  "I couldn't find enough evidence in the repository."

Retrieved Context:
{context}

Question:
{question}

Return your answer in this format:

### Answer
<answer>

### Supporting Sources
List every SOURCE used in your answer.
- Commit abc123
- PR #1234
- Issue #81
""")

In [65]:
def format_docs(docs):
    formatted = []

    for doc in docs:
        source = ""

        if doc.metadata["type"] == "issue":
            source = f"Issue #{doc.metadata['number']}"

        elif doc.metadata["type"] == "pull_request":
            source = f"PR #{doc.metadata['number']}"

        elif doc.metadata["type"] == "commit":
            source = f"Commit {doc.metadata['sha'][:7]}"

        formatted.append(
            f"""SOURCE: {source}

{doc.page_content}
"""
        )

    return "\n\n".join(formatted)

In [66]:
output_parser = StrOutputParser()

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": lambda x: x
    }
    | prompt
    | llm
    | output_parser
)

In [67]:
question = "Why was dependency injection introduced in FastAPI?"

response = rag_chain.invoke(question)

print(response)

### Answer
Based on the retrieved repository context, dependency passing (dependency injection) is advocated and used in FastAPI to:
- **Modularize an application**: It allows developers to structure their application cleanly rather than relying on globally accessible module variables (such as database repository objects).
- **Govern access**: It helps control and manage access within the application.
- **Facilitate testing**: It makes testing easier by allowing dependencies to be passed and mocked/replaced.
- **Manage security and arguments**: It allows for security-related requirements to be handled as dependencies, which avoids unused argument warnings in functions and prevents the accidental removal of critical security parameters by developers unfamiliar with the code.

### Supporting Sources
- Issue #81
- Issue #171


Evaluation using **RAGAs** was skipped due to library compatibility issues with the latest LangChain version and exhausted Gemini API quota.

The PatchContext RAG pipeline was successfully tested manually and produces grounded answers with relevant source citations.

In [92]:
import shutil

shutil.make_archive("faiss_index", "zip", "faiss_index")

'/root/fastapi/faiss_index.zip'

In [93]:
from google.colab import files

files.download("faiss_index.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>